In [19]:
from ultralytics import YOLO
import pandas as pd
import os
import cv2

from models import CNN_Model, LSTM_Model, BiLSTM_Model, Rule_Model

# GENERATE keypoints here
model = YOLO("yolo11n-pose.pt")  # load an official model

In [3]:
# Predict with the model
results = model("bus.jpg")  # predict on an image


image 1/1 C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3\DL\Assignment\ass 3\bus.jpg: 640x480 4 persons, 83.4ms
Speed: 8.8ms preprocess, 83.4ms inference, 243.2ms postprocess per image at shape (1, 3, 640, 480)


In [7]:
len(results[0])

4

In [10]:
import numpy as np
from ultralytics import YOLO
import pandas as pd
import os
import cv2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Define model parameters
input_size = 51   # Number of features (e.g., in your falling data)
hidden_size = 16  # Number of units in the hidden state
num_layers = 1    # Number of LSTM layers (you can experiment with this)
dropout = 0.5     # Dropout rate for regularization

# Initialize the LSTM_Model model
lstm_model = LSTM_Model(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout)
lstm_model.load_state_dict(torch.load('./model/lstm_model.pth'))

# Set the model to evaluation mode (important for inference)
lstm_model.eval()
lstm_model.to(device)

LSTM_Model(
  (lstm1): LSTM(51, 16, batch_first=True)
  (lstm2): LSTM(16, 16, batch_first=True)
  (fc): Linear(in_features=16, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [32]:
import os
import cv2
import torch

def is_none_or_empty(x):
    return x is None or len(x) == 0

def process_video_with_model(yolo_model, dl_model, device, input_path="./fall_dataset/processed/fall-01-cam0.mp4", output_path='lstm_annotated_output.mp4'):
    all_keypoints = []

    cap = cv2.VideoCapture(input_path)

    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Set up the output writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print(f"Processing {input_path}")
    results = yolo_model.track(source=input_path)

    max_pred = 0.0
    for frame_number, result in enumerate(results):
        frame = result.orig_img.copy()
        boxes = result.boxes
        keypoints = result.keypoints

        if is_none_or_empty(boxes) or is_none_or_empty(keypoints):
            continue

        best_index = None
        best_conf = 0.0

        for i in range(len(boxes)):
            conf = float(boxes.conf[i])
            cls = int(boxes.cls[i])
            if cls == 0 and conf > best_conf:
                best_conf = conf
                best_index = i

        if best_index is None:
            continue

        keypoints_tensor = keypoints.data[best_index]
        keypoints_np = keypoints_tensor.cpu().detach().numpy()
        flat = keypoints_np.flatten().tolist()

        for i in range(0, 51, 3):
            flat[i] /= width   # normalize x
            flat[i + 1] /= height  # normalize y

        input_tensor = torch.tensor(flat, dtype=torch.float32).view(1, 1, 51).to(device)

        with torch.no_grad():
            output = dl_model(input_tensor)

        prediction_value = output.item()
        max_pred = max(max_pred, prediction_value)
        prediction_label = "FALL" if prediction_value > 0.5 else "SAFE"

        x, y = int(keypoints_np[0][0]), int(keypoints_np[0][1])
        color = (0, 0, 255) if prediction_label == "FALL" else (0, 255, 0)
        cv2.putText(frame, prediction_label, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        out.write(frame)
        print(f"Frame {frame_number}: Prediction = {prediction_label} ({prediction_value:.4f})")

    out.release()
    print(f"Saved annotated video to: {output_path}")
    print("max_pred", max_pred)


In [33]:
process_video_with_model(yolo_model=model, dl_model=lstm_model, device=device)

Processing ./fall_dataset/processed/fall-01-cam0.mp4


WARNING  inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/160) C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3\DL\Assignment\ass 3\fall_dataset\processed\fall-01-cam0.mp4: 480x640 1 person, 35.2ms
video 1/1 (frame 2/160) C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3\DL\Assignment\ass 3\fall_dataset\processed\fall-01-cam0.mp4: 480x640 1 person, 34.1ms
video 1/1 (frame 3/160) C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3

In [17]:
import numpy as np
from ultralytics import YOLO
import pandas as pd
import os
import cv2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize the LSTM_Model model
cnn_model = CNN_Model()
cnn_model.load_state_dict(torch.load('./model/cnn_model.pth'))

# Set the model to evaluation mode (important for inference)
cnn_model.eval()
cnn_model.to(device)

CNN_Model(
  (conv1): Conv1d(1, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): AdaptiveMaxPool1d(output_size=1)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [34]:
process_video_with_model(yolo_model=model, dl_model=cnn_model, device=device, output_path='cnn_annotated_output.mp4')

Processing ./fall_dataset/processed/fall-01-cam0.mp4


WARNING  inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/160) C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3\DL\Assignment\ass 3\fall_dataset\processed\fall-01-cam0.mp4: 480x640 1 person, 37.3ms
video 1/1 (frame 2/160) C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3\DL\Assignment\ass 3\fall_dataset\processed\fall-01-cam0.mp4: 480x640 1 person, 35.2ms
video 1/1 (frame 3/160) C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3

In [20]:
# cv2.putText() is another OpenCV function used to add text to an image.
# image: The image to which the text will be added.
# 'Person Fell down': The text message that will be displayed on the image.
# (11, 100): The position of the text (top-left corner of the text box) in (x, y) coordinates. This is the point where the text starts to be drawn on the image.
# 0: The font type (0 corresponds to the default font in OpenCV).
# 1: The font scale, i.e., the size of the text. A scale of 1 means the text will be the default size.
# [0, 0, 2550]: The color of the text. It uses the BGR format (Blue, Green, Red). This is a red color with a high intensity (2550 is likely an error and should be something like [0, 0, 255]).
# thickness=3: The thickness of the text.
# lineType=cv2.LINE_AA: This ensures that the text is drawn with antialiased lines for smoother edges.
def falling_alarm(image, bbox):
    x_min, y_min, x_max, y_max = bbox
    cv2.rectangle(image, (int(x_min), int(y_min)), (int(x_max), int(y_max)), color=(0, 0, 255),
                  thickness=5, lineType=cv2.LINE_AA)
    cv2.putText(image, 'Person Fell down', (11, 100), 0, 1, [0, 0, 2550], thickness=3, lineType=cv2.LINE_AA)

In [26]:
# Predict with the model

input_folder = './fall_dataset/processed'

all_keypoints = []

def is_none_or_empty(x):
    return x is None or len(x) == 0

cap = cv2.VideoCapture("./fall_dataset/processed/fall-01-cam0.mp4")

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Set up the output writer
output_path = 'rule_annotated_output.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

for filename in os.listdir(input_folder):
    filename = "fall-01-cam0.mp4"
    if filename.endswith('.mp4'):
        full_path = os.path.join(input_folder, filename)
        print("Processing {}".format(full_path))
        results = model.track(source=full_path)

        max_pred = 0.0
        for frame_number, result in enumerate(results):  # each is a frame
            is_fall, bbox = Rule_Model(result, width, height)
            frame = result.orig_img.copy()
            
            if is_fall:
                falling_alarm(frame, bbox) 
            out.write(frame)
            print("pred", is_fall)

    print("max_pred", max_pred)
    break # remove this to process more vids


out.release()
print("Saved annotated video to:", output_path)

Processing ./fall_dataset/processed\fall-01-cam0.mp4


WARNING  inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/160) C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3\DL\Assignment\ass 3\fall_dataset\processed\fall-01-cam0.mp4: 480x640 1 person, 35.4ms
video 1/1 (frame 2/160) C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3\DL\Assignment\ass 3\fall_dataset\processed\fall-01-cam0.mp4: 480x640 1 person, 34.7ms
video 1/1 (frame 3/160) C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3

In [35]:
import os
import cv2
import torch

def process_video_ensemble_model(yolo_model, device, input_path="./fall_dataset/processed/fall-01-cam0.mp4", output_path='lstm_annotated_output.mp4'):
    all_keypoints = []

    cap = cv2.VideoCapture(input_path)

    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Set up the output writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print(f"Processing {input_path}")
    results = yolo_model.track(source=input_path)

    max_pred = 0.0
    for frame_number, result in enumerate(results):
        frame = result.orig_img.copy()
        boxes = result.boxes
        keypoints = result.keypoints

        if is_none_or_empty(boxes) or is_none_or_empty(keypoints):
            continue

        best_index = None
        best_conf = 0.0

        for i in range(len(boxes)):
            conf = float(boxes.conf[i])
            cls = int(boxes.cls[i])
            if cls == 0 and conf > best_conf:
                best_conf = conf
                best_index = i

        if best_index is None:
            continue

        keypoints_tensor = keypoints.data[best_index]
        keypoints_np = keypoints_tensor.cpu().detach().numpy()
        flat = keypoints_np.flatten().tolist()

        for i in range(0, 51, 3):
            flat[i] /= width   # normalize x
            flat[i + 1] /= height  # normalize y

        input_tensor = torch.tensor(flat, dtype=torch.float32).view(1, 1, 51).to(device)

        with torch.no_grad():
            cnn_pred = cnn_model(input_tensor).item()
            lstm_pred = lstm_model(input_tensor).item()

        rule_pred_bool, bbox = Rule_Model(result, width, height)
        rule_pred = float(rule_pred_bool)

        prediction_value = (cnn_pred + lstm_pred + rule_pred) / 3.0
        max_pred = max(max_pred, prediction_value)
        prediction_label = "FALL" if prediction_value > 0.5 else "SAFE"

        x, y = int(keypoints_np[0][0]), int(keypoints_np[0][1])
        color = (0, 0, 255) if prediction_label == "FALL" else (0, 255, 0)
        cv2.putText(frame, prediction_label, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        out.write(frame)
        print(f"Frame {frame_number}: Prediction = {prediction_label} ({prediction_value:.4f})")

    out.release()
    print(f"Saved annotated video to: {output_path}")
    print("max_pred", max_pred)


In [37]:
process_video_ensemble_model(yolo_model=model, device=device, output_path='ensemble_annotated_output.mp4')

Processing ./fall_dataset/processed/fall-01-cam0.mp4


WARNING  inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/160) C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3\DL\Assignment\ass 3\fall_dataset\processed\fall-01-cam0.mp4: 480x640 1 person, 35.2ms
video 1/1 (frame 2/160) C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3\DL\Assignment\ass 3\fall_dataset\processed\fall-01-cam0.mp4: 480x640 1 person, 34.3ms
video 1/1 (frame 3/160) C:\Users\sonnpm\OneDrive - UTS\Desktop\UTS\Semester 3